# 🧠 Parkinson's Disease Detection via Spiral & Wave Drawing Analysis
### Deep Learning + Classical Feature Engineering Pipeline
---

## 📦 Cell 1: Install & Import Dependencies

In [1]:
# Install required packages (run once)
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['opencv-python-headless', 'scikit-image', 'tensorflow', 'grad-cam',
            'seaborn', 'imutils', 'joblib', 'scikit-learn', 'matplotlib', 'numpy', 'Pillow']:
    install(pkg)

print('✅ All packages installed')

✅ All packages installed


In [2]:
import os, cv2, joblib, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from pathlib import Path
from PIL import Image
from glob import glob
from tqdm.notebook import tqdm

# Scikit-learn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, precision_score,
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, matthews_corrcoef
)

# Scipy
from scipy import ndimage, signal
from scipy.stats import entropy

# Skimage
from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from skimage import filters, morphology
from skimage.measure import label, regionprops

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')
np.random.seed(42)
tf.random.set_seed(42)

print(f'✅ TensorFlow {tf.__version__} | NumPy {np.__version__}')

✅ TensorFlow 2.21.0 | NumPy 2.4.3


## 📁 Cell 2: Configuration & Output Directory Setup

In [5]:
# ─── CONFIGURATION ────────────────────────────────────────────────────────────
BASE_DATA_DIR = Path('drawings')          
OUTPUT_DIR    = Path('outputs')
MODEL_DIR     = OUTPUT_DIR / 'models'
PLOT_DIR      = OUTPUT_DIR / 'plots'
FEATURE_DIR   = OUTPUT_DIR / 'features'

IMG_SIZE      = (128, 128)
BATCH_SIZE    = 16
EPOCHS        = 30
DRAWING_TYPES = ['spiral', 'wave']

# Create directories
for d in [OUTPUT_DIR, MODEL_DIR, PLOT_DIR, FEATURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('📂 Output directories created:')
for d in [OUTPUT_DIR, MODEL_DIR, PLOT_DIR, FEATURE_DIR]:
    print(f'   {d}')

# ─── DATASET PATH BUILDER ─────────────────────────────────────────────────────
def get_paths(dtype):
    """Return dict of all image paths for a drawing type."""
    root = BASE_DATA_DIR / dtype
    return {
        'train_healthy':   list((root / 'training' / 'healthy').glob('*.*')),
        'train_parkinson': list((root / 'training' / 'parkinson').glob('*.*')),
        'test_healthy':    list((root / 'testing'  / 'healthy').glob('*.*')),
        'test_parkinson':  list((root / 'testing'  / 'parkinson').glob('*.*')),
    }

for dt in DRAWING_TYPES:
    p = get_paths(dt)
    print(f'\n{dt.upper()}:')
    for k, v in p.items():
        print(f'   {k}: {len(v)} images')

📂 Output directories created:
   outputs
   outputs\models
   outputs\plots
   outputs\features

SPIRAL:
   train_healthy: 36 images
   train_parkinson: 36 images
   test_healthy: 15 images
   test_parkinson: 15 images

WAVE:
   train_healthy: 36 images
   train_parkinson: 36 images
   test_healthy: 15 images
   test_parkinson: 15 images


## 🖼️ Cell 3: Preprocessing Pipeline

In [6]:
class DrawingPreprocessor:
    """Full preprocessing pipeline for spiral/wave drawings."""

    def __init__(self, target_size=(128, 128)):
        self.target_size = target_size

    def load(self, path):
        img = cv2.imread(str(path))
        if img is None:
            raise FileNotFoundError(f'Cannot load: {path}')
        return img

    def to_gray(self, img):
        return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    def resize(self, img):
        return cv2.resize(img, self.target_size, interpolation=cv2.INTER_AREA)

    def enhance_contrast(self, gray):
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        return clahe.apply(gray)

    def denoise(self, gray):
        return cv2.fastNlMeansDenoising(gray, h=10)

    def edge_detect(self, gray):
        blurred = cv2.GaussianBlur(gray, (5,5), 0)
        edges   = cv2.Canny(blurred, threshold1=30, threshold2=100)
        return edges

    def extract_contours(self, edges):
        contours, _ = cv2.findContours(
            edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        # Sort by area descending
        return sorted(contours, key=cv2.contourArea, reverse=True)

    def process(self, path):
        """Full pipeline → returns dict of intermediate images."""
        img      = self.load(path)
        gray     = self.to_gray(img)
        gray_r   = self.resize(gray)
        enhanced = self.enhance_contrast(gray_r)
        denoised = self.denoise(enhanced)
        edges    = self.edge_detect(denoised)
        contours = self.extract_contours(edges)

        # Draw contours on blank canvas
        canvas   = np.zeros_like(denoised)
        cv2.drawContours(canvas, contours, -1, 255, 1)

        return {
            'original': self.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)),
            'gray':     gray_r,
            'enhanced': enhanced,
            'denoised': denoised,
            'edges':    edges,
            'contours': canvas,
            'contour_list': contours,
        }

    def normalize_for_cnn(self, path):
        """Return (H,W,3) float32 array normalized [0,1]."""
        res = self.process(path)
        rgb = np.stack([res['denoised']]*3, axis=-1).astype(np.float32) / 255.0
        return rgb


preproc = DrawingPreprocessor(target_size=IMG_SIZE)
print('✅ DrawingPreprocessor ready')

✅ DrawingPreprocessor ready


## 🔍 Cell 4: Visualise Preprocessing Steps

In [7]:
def visualise_preprocessing(dtype='spiral'):
    paths = get_paths(dtype)
    samples = [
        (paths['train_healthy'][0],   'Healthy'),
        (paths['train_parkinson'][0], 'Parkinson'),
    ]

    stages = ['original','gray','enhanced','denoised','edges','contours']
    titles = ['Original','Grayscale','CLAHE Enhanced','Denoised','Canny Edges','Contours']
    cmaps  = ['viridis','gray','gray','gray','gray','gray']

    fig, axes = plt.subplots(2, 6, figsize=(22, 8))
    fig.suptitle(f'Preprocessing Pipeline — {dtype.upper()} Drawings',
                 fontsize=16, fontweight='bold', y=1.01)

    for row, (path, label_name) in enumerate(samples):
        result = preproc.process(path)
        for col, (stage, title, cmap) in enumerate(zip(stages, titles, cmaps)):
            ax = axes[row][col]
            ax.imshow(result[stage], cmap=cmap)
            if row == 0:
                ax.set_title(title, fontsize=11, fontweight='bold')
            ax.set_ylabel(label_name if col == 0 else '', fontsize=11)
            ax.axis('off')

    plt.tight_layout()
    save_path = PLOT_DIR / f'preprocessing_{dtype}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 Saved → {save_path}')

for dt in DRAWING_TYPES:
    visualise_preprocessing(dt)

💾 Saved → outputs\plots\preprocessing_spiral.png
💾 Saved → outputs\plots\preprocessing_wave.png


## 📐 Cell 5: Feature Extraction

In [8]:
class FeatureExtractor:
    """Extract hand-crafted features from preprocessed drawings."""

    def __init__(self):
        self.feature_names = []

    # ── 1. Tremor Frequency ──────────────────────────────────────────────────
    def tremor_frequency(self, gray):
        """Estimate dominant tremor freq via FFT of row-wise intensity profile."""
        profile  = gray.mean(axis=1).astype(np.float32)
        fft_vals = np.abs(np.fft.rfft(profile - profile.mean()))
        freqs    = np.fft.rfftfreq(len(profile))
        dom_freq = freqs[np.argmax(fft_vals[1:]) + 1]   # skip DC
        spectral_entropy = entropy(fft_vals + 1e-9)
        total_power = np.sum(fft_vals**2)
        high_freq_power = np.sum(fft_vals[len(fft_vals)//4:]**2)
        tremor_index = high_freq_power / (total_power + 1e-9)
        return [dom_freq, spectral_entropy, tremor_index]

    # ── 2. Line Smoothness ───────────────────────────────────────────────────
    def line_smoothness(self, edges, contours):
        """Curvature-based smoothness of the longest contour."""
        if not contours:
            return [0.0, 0.0, 0.0]
        cnt = contours[0]
        if len(cnt) < 5:
            return [0.0, 0.0, 0.0]
        pts = cnt.squeeze().astype(float)
        # Curvature
        dx  = np.gradient(pts[:, 0])
        dy  = np.gradient(pts[:, 1])
        d2x = np.gradient(dx)
        d2y = np.gradient(dy)
        curv = np.abs(dx * d2y - dy * d2x) / (dx**2 + dy**2 + 1e-9)**1.5
        mean_curv  = np.mean(curv)
        std_curv   = np.std(curv)
        max_curv   = np.max(curv)
        return [mean_curv, std_curv, max_curv]

    # ── 3. Deviation from Ideal Spiral ───────────────────────────────────────
    def spiral_deviation(self, gray):
        """Compare edge distribution to ideal Archimedean spiral template."""
        h, w = gray.shape
        cx, cy = w // 2, h // 2

        # Generate ideal spiral
        theta    = np.linspace(0, 6*np.pi, 500)
        r_max    = min(cx, cy) * 0.9
        r_spiral = r_max * theta / (6 * np.pi)
        xs = (cx + r_spiral * np.cos(theta)).astype(int).clip(0, w-1)
        ys = (cy + r_spiral * np.sin(theta)).astype(int).clip(0, h-1)

        ideal = np.zeros((h, w), dtype=np.uint8)
        ideal[ys, xs] = 255

        # Distance transform
        dt_ideal  = ndimage.distance_transform_edt(255 - ideal)
        edges_bin = (gray > 50).astype(np.uint8)
        masked    = dt_ideal * edges_bin

        mean_dev  = float(masked[masked > 0].mean()) if masked.any() else 0.0
        std_dev   = float(masked[masked > 0].std())  if masked.any() else 0.0

        # Circularity deviation
        ys_e, xs_e = np.where(edges_bin > 0)
        if len(ys_e) > 10:
            dists = np.sqrt((xs_e - cx)**2 + (ys_e - cy)**2)
            circ_dev = np.std(dists) / (np.mean(dists) + 1e-9)
        else:
            circ_dev = 0.0

        return [mean_dev, std_dev, circ_dev]

    # ── 4. Fractal Dimension (Box-counting) ───────────────────────────────────
    def fractal_dimension(self, binary):
        """Box-counting fractal dimension of binary edge image."""
        def box_count(img, box_size):
            h, w = img.shape
            n_rows = int(np.ceil(h / box_size))
            n_cols = int(np.ceil(w / box_size))
            count = 0
            for r in range(n_rows):
                for c in range(n_cols):
                    patch = img[r*box_size:(r+1)*box_size,
                                c*box_size:(c+1)*box_size]
                    if patch.any():
                        count += 1
            return count

        binary_bin = (binary > 0).astype(np.uint8)
        sizes = [2, 4, 8, 16, 32]
        counts = [box_count(binary_bin, s) for s in sizes]
        valid = [(s, c) for s, c in zip(sizes, counts) if c > 0]
        if len(valid) < 2:
            return [0.0]
        log_s = np.log([v[0] for v in valid])
        log_c = np.log([v[1] for v in valid])
        coeffs = np.polyfit(log_s, log_c, 1)
        return [abs(coeffs[0])]   # fractal dimension

    # ── 5. Texture (LBP + GLCM) ──────────────────────────────────────────────
    def texture_features(self, gray):
        lbp   = local_binary_pattern(gray, P=8, R=1, method='uniform')
        hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0,10), density=True)

        glcm  = graycomatrix(gray, [1], [0, np.pi/2], levels=256,
                              symmetric=True, normed=True)
        contrast    = graycoprops(glcm, 'contrast').mean()
        dissim      = graycoprops(glcm, 'dissimilarity').mean()
        homogeneity = graycoprops(glcm, 'homogeneity').mean()
        energy      = graycoprops(glcm, 'energy').mean()
        correlation = graycoprops(glcm, 'correlation').mean()
        return list(hist) + [contrast, dissim, homogeneity, energy, correlation]

    # ── 6. Statistical Moments ────────────────────────────────────────────────
    def statistical_moments(self, gray):
        flat = gray.astype(float).ravel()
        from scipy.stats import skew, kurtosis
        return [
            np.mean(flat), np.std(flat), skew(flat), kurtosis(flat),
            np.percentile(flat, 25), np.percentile(flat, 75)
        ]

    # ── Master extract ────────────────────────────────────────────────────────
    def extract(self, path):
        result = preproc.process(path)
        gray   = result['denoised']
        edges  = result['edges']
        contours = result['contour_list']

        feats = []
        feats += self.tremor_frequency(gray)          # 3
        feats += self.line_smoothness(edges, contours) # 3
        feats += self.spiral_deviation(edges)          # 3
        feats += self.fractal_dimension(edges)         # 1
        feats += self.texture_features(gray)           # 15
        feats += self.statistical_moments(gray)        # 6
        return np.array(feats, dtype=np.float32)

    @property
    def n_features(self):
        return 3+3+3+1+15+6  # = 31


feat_extractor = FeatureExtractor()
print(f'✅ FeatureExtractor ready | {feat_extractor.n_features} features per image')

✅ FeatureExtractor ready | 31 features per image


## 📊 Cell 6: Build Feature Datasets

In [9]:
def build_feature_dataset(dtype):
    """Extract features for all images and return train/test splits."""
    paths = get_paths(dtype)

    def load_split(healthy_paths, parkinson_paths):
        X, y = [], []
        for p in healthy_paths:
            try:
                X.append(feat_extractor.extract(p))
                y.append(0)  # healthy
            except Exception as e:
                print(f'  ⚠ Skip {p.name}: {e}')
        for p in parkinson_paths:
            try:
                X.append(feat_extractor.extract(p))
                y.append(1)  # parkinson
            except Exception as e:
                print(f'  ⚠ Skip {p.name}: {e}')
        return np.array(X), np.array(y)

    print(f'\nExtracting features for {dtype.upper()}...')
    X_train, y_train = load_split(paths['train_healthy'], paths['train_parkinson'])
    X_test,  y_test  = load_split(paths['test_healthy'],  paths['test_parkinson'])

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    # Save scaler
    joblib.dump(scaler, FEATURE_DIR / f'scaler_{dtype}.pkl')
    np.save(FEATURE_DIR / f'X_train_{dtype}.npy', X_train)
    np.save(FEATURE_DIR / f'y_train_{dtype}.npy', y_train)
    np.save(FEATURE_DIR / f'X_test_{dtype}.npy',  X_test)
    np.save(FEATURE_DIR / f'y_test_{dtype}.npy',  y_test)

    print(f'  Train: {X_train.shape} | Test: {X_test.shape}')
    return X_train, y_train, X_test, y_test, scaler


feature_data = {}
for dt in DRAWING_TYPES:
    feature_data[dt] = build_feature_dataset(dt)

print('\n✅ Feature extraction complete')


Extracting features for SPIRAL...
  Train: (72, 31) | Test: (30, 31)

Extracting features for WAVE...
  Train: (72, 31) | Test: (30, 31)

✅ Feature extraction complete


## 📈 Cell 7: Feature Importance & Visualisation

In [10]:
FEATURE_LABELS = (
    ['TremorFreq','SpectralEntropy','TremorIndex']
    + ['MeanCurvature','StdCurvature','MaxCurvature']
    + ['SpiralDevMean','SpiralDevStd','CircularityDev']
    + ['FractalDim']
    + [f'LBP_{i}' for i in range(10)]
    + ['GLCM_Contrast','GLCM_Dissim','GLCM_Homogeneity','GLCM_Energy','GLCM_Corr']
    + ['Mean','Std','Skewness','Kurtosis','Q25','Q75']
)

def plot_feature_distributions(dtype):
    X_train, y_train, _, _, _ = feature_data[dtype]

    # Select top 12 features by t-test separation
    from scipy.stats import ttest_ind
    X_h = X_train[y_train == 0]
    X_p = X_train[y_train == 1]
    _, pvals = ttest_ind(X_h, X_p, axis=0)
    top12 = np.argsort(pvals)[:12]

    fig, axes = plt.subplots(3, 4, figsize=(18, 10))
    fig.suptitle(f'Top-12 Discriminative Features — {dtype.upper()}',
                 fontsize=14, fontweight='bold')

    for i, idx in enumerate(top12):
        ax = axes[i//4][i%4]
        ax.hist(X_h[:, idx], bins=12, alpha=0.6, color='steelblue', label='Healthy')
        ax.hist(X_p[:, idx], bins=12, alpha=0.6, color='tomato',    label='Parkinson')
        ax.set_title(FEATURE_LABELS[idx], fontsize=9, fontweight='bold')
        ax.set_xlabel('Normalised Value', fontsize=8)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = PLOT_DIR / f'feature_distributions_{dtype}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 Saved → {save_path}')

for dt in DRAWING_TYPES:
    plot_feature_distributions(dt)

💾 Saved → outputs\plots\feature_distributions_spiral.png
💾 Saved → outputs\plots\feature_distributions_wave.png


## 🤖 Cell 8: Classical ML Classifiers (SVM + Random Forest)

In [11]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

def evaluate_classical_models(dtype):
    X_train, y_train, X_test, y_test, _ = feature_data[dtype]

    models = {
        'SVM (RBF)':    SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    }

    results = {}
    for name, clf in models.items():
        clf.fit(X_train, y_train)
        y_pred  = clf.predict(X_test)
        y_proba = clf.predict_proba(X_test)[:, 1]

        results[name] = {
            'clf':       clf,
            'y_pred':    y_pred,
            'y_proba':   y_proba,
            'accuracy':  accuracy_score(y_test, y_pred),
            'f1':        f1_score(y_test, y_pred),
            'recall':    recall_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred),
            'roc_auc':   roc_auc_score(y_test, y_proba),
            'mcc':       matthews_corrcoef(y_test, y_pred),
        }

        # Save model
        joblib.dump(clf, MODEL_DIR / f'{name.split()[0].lower()}_{dtype}.pkl')

    return results, y_test


classical_results = {}
for dt in DRAWING_TYPES:
    print(f'\n─── {dt.upper()} ───')
    res, y_test = evaluate_classical_models(dt)
    classical_results[dt] = (res, y_test)
    for name, metrics in res.items():
        print(f'  {name:<20} Acc={metrics["accuracy"]:.3f} | '
              f'F1={metrics["f1"]:.3f} | AUC={metrics["roc_auc"]:.3f}')

print('\n✅ Classical models trained & saved')


─── SPIRAL ───
  SVM (RBF)            Acc=0.733 | F1=0.714 | AUC=0.880
  Random Forest        Acc=0.833 | F1=0.839 | AUC=0.911

─── WAVE ───
  SVM (RBF)            Acc=0.733 | F1=0.765 | AUC=0.893
  Random Forest        Acc=0.700 | F1=0.710 | AUC=0.849

✅ Classical models trained & saved


## 🧪 Cell 9: Confusion Matrices — Classical Models

In [12]:
def plot_confusion_matrices(dtype):
    res, y_test = classical_results[dtype]
    model_names = list(res.keys())

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'Confusion Matrices — {dtype.upper()} Drawings',
                 fontsize=14, fontweight='bold')

    for ax, name in zip(axes, model_names):
        cm = confusion_matrix(y_test, res[name]['y_pred'])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Healthy','Parkinson'],
                    yticklabels=['Healthy','Parkinson'],
                    linewidths=1, linecolor='white')
        ax.set_title(f'{name}\nAcc={res[name]["accuracy"]:.3f}', fontweight='bold')
        ax.set_xlabel('Predicted', fontsize=10)
        ax.set_ylabel('True', fontsize=10)

    plt.tight_layout()
    save_path = PLOT_DIR / f'confusion_matrix_classical_{dtype}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 Saved → {save_path}')

for dt in DRAWING_TYPES:
    plot_confusion_matrices(dt)

💾 Saved → outputs\plots\confusion_matrix_classical_spiral.png
💾 Saved → outputs\plots\confusion_matrix_classical_wave.png


## 📉 Cell 10: ROC-AUC Curves — Classical Models

In [13]:
COLORS = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']

def plot_roc_curves_classical(dtype):
    res, y_test = classical_results[dtype]

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.plot([0,1],[0,1],'k--', lw=1, label='Random Baseline')

    for (name, metrics), color in zip(res.items(), COLORS):
        fpr, tpr, _ = roc_curve(y_test, metrics['y_proba'])
        auc_val = metrics['roc_auc']
        ax.plot(fpr, tpr, lw=2.5, color=color,
                label=f'{name} (AUC={auc_val:.3f})')

    ax.fill_between([0,1],[0,1], alpha=0.05, color='gray')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(f'ROC-AUC Curves — {dtype.upper()} (Classical Models)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=11, loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])

    save_path = PLOT_DIR / f'roc_auc_classical_{dtype}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 Saved → {save_path}')

for dt in DRAWING_TYPES:
    plot_roc_curves_classical(dt)

💾 Saved → outputs\plots\roc_auc_classical_spiral.png
💾 Saved → outputs\plots\roc_auc_classical_wave.png


## 🧠 Cell 11: CNN Architecture

In [14]:
def build_cnn(input_shape=(128, 128, 3)):
    """Custom CNN classifier for drawing analysis."""
    inp = layers.Input(shape=input_shape, name='input')

    # Block 1
    x = layers.Conv2D(32, 3, padding='same', activation='relu', name='conv1a')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu', name='conv1b')(x)
    x = layers.MaxPooling2D(2, name='pool1')(x)
    x = layers.Dropout(0.25)(x)

    # Block 2
    x = layers.Conv2D(64, 3, padding='same', activation='relu', name='conv2a')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu', name='conv2b')(x)
    x = layers.MaxPooling2D(2, name='pool2')(x)
    x = layers.Dropout(0.25)(x)

    # Block 3
    x = layers.Conv2D(128, 3, padding='same', activation='relu', name='conv3a')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, 3, padding='same', activation='relu', name='conv3b')(x)
    x = layers.MaxPooling2D(2, name='pool3')(x)
    x = layers.Dropout(0.35)(x)

    # Head
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dense(256, activation='relu', name='fc1')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation='sigmoid', name='output')(x)

    model = Model(inp, out, name='PD_CNN')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    return model


model_test = build_cnn()
model_test.summary()
print('\n✅ CNN architecture defined')

Model: "PD_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1a (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1b (Conv2D)                 │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2a (Conv2D)                 │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2b (Conv2D)                 │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3a (Conv2D)                 │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3b (Conv2D)                 │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool3 (MaxPooling2D)            │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling2D)    │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321,185 (1.23 MB)

 Trainable params: 320,737 (1.22 MB)

 Non-trainable params: 448 (1.75 KB)


✅ CNN architecture defined


## 📂 Cell 12: CNN Data Pipeline

In [15]:
def build_cnn_arrays(dtype):
    """Load all images as numpy arrays with labels."""
    paths = get_paths(dtype)

    def load_images(path_list, label_val):
        imgs, lbls = [], []
        for p in path_list:
            try:
                arr = preproc.normalize_for_cnn(p)
                imgs.append(arr)
                lbls.append(label_val)
            except Exception as e:
                print(f'  ⚠ {p.name}: {e}')
        return imgs, lbls

    print(f'Loading CNN arrays for {dtype.upper()}...')
    tr_h_X, tr_h_y = load_images(paths['train_healthy'],   0)
    tr_p_X, tr_p_y = load_images(paths['train_parkinson'], 1)
    te_h_X, te_h_y = load_images(paths['test_healthy'],    0)
    te_p_X, te_p_y = load_images(paths['test_parkinson'],  1)

    X_train = np.array(tr_h_X + tr_p_X)
    y_train = np.array(tr_h_y + tr_p_y)
    X_test  = np.array(te_h_X + te_p_X)
    y_test  = np.array(te_h_y + te_p_y)

    # Shuffle training
    idx = np.random.permutation(len(X_train))
    X_train, y_train = X_train[idx], y_train[idx]

    print(f'  Train: {X_train.shape} | Test: {X_test.shape}')
    return X_train, y_train, X_test, y_test


cnn_data = {}
for dt in DRAWING_TYPES:
    cnn_data[dt] = build_cnn_arrays(dt)

print('\n✅ CNN data ready')

Loading CNN arrays for SPIRAL...
  Train: (72, 128, 128, 3) | Test: (30, 128, 128, 3)
Loading CNN arrays for WAVE...
  Train: (72, 128, 128, 3) | Test: (30, 128, 128, 3)

✅ CNN data ready


## 🏋️ Cell 13: CNN Training

In [16]:
# Augmentation
def get_augmenter():
    return ImageDataGenerator(
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        horizontal_flip=True,
        zoom_range=0.1,
    )


def train_cnn(dtype):
    X_train, y_train, X_test, y_test = cnn_data[dtype]

    model = build_cnn(input_shape=(*IMG_SIZE, 3))

    callbacks = [
        EarlyStopping(patience=10, restore_best_weights=True, monitor='val_auc', mode='max'),
        ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-6, monitor='val_auc', mode='max'),
        ModelCheckpoint(
            str(MODEL_DIR / f'cnn_{dtype}_best.keras'),
            monitor='val_auc', mode='max', save_best_only=True
        ),
    ]

    aug = get_augmenter()
    aug_gen = aug.flow(X_train, y_train, batch_size=BATCH_SIZE)

    print(f'\n🏋️  Training CNN for {dtype.upper()}...')
    history = model.fit(
        aug_gen,
        steps_per_epoch=max(1, len(X_train) // BATCH_SIZE),
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=callbacks,
        verbose=1,
    )

    # Also save as .h5
    model.save(MODEL_DIR / f'cnn_{dtype}.h5')

    return model, history, X_test, y_test


cnn_models = {}
cnn_histories = {}
cnn_tests = {}

for dt in DRAWING_TYPES:
    model, history, X_test, y_test = train_cnn(dt)
    cnn_models[dt]   = model
    cnn_histories[dt] = history
    cnn_tests[dt]    = (X_test, y_test)

print('\n✅ CNN training complete — models saved')


🏋️  Training CNN for SPIRAL...
Epoch 1/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 22s 2s/step - accuracy: 0.5000 - auc: 0.4722 - loss: 0.8452 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.6932 - learning_rate: 0.0010
Epoch 2/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 197ms/step - accuracy: 0.1250 - auc: 0.1875 - loss: 0.9649 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.6933 - learning_rate: 0.0010
Epoch 3/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step - accuracy: 0.5000 - auc: 0.4840 - loss: 0.8095 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.6933 - learning_rate: 0.0010
Epoch 4/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 194ms/step - accuracy: 0.5000 - auc: 0.7302 - loss: 0.6733 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.6933 - learning_rate: 0.0010
Epoch 5/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.5714 - auc: 0.5306 - loss: 0.7982 - val_accuracy: 0.5000 - val_auc: 0.4667 - val_loss: 0.6945 - learning_rate: 0.0010
Epoch 6/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 232ms/step - accuracy: 0.75


🏋️  Training CNN for WAVE...
Epoch 1/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 14s 2s/step - accuracy: 0.5179 - auc: 0.4335 - loss: 0.8164 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.6934 - learning_rate: 0.0010
Epoch 2/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 263ms/step - accuracy: 0.6250 - auc: 0.7455 - loss: 0.5849 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.6935 - learning_rate: 0.0010
Epoch 3/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 2s/step - accuracy: 0.6250 - auc: 0.6703 - loss: 0.6830 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.6964 - learning_rate: 0.0010
Epoch 4/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 366ms/step - accuracy: 0.4375 - auc: 0.6273 - loss: 0.7845 - val_accuracy: 0.5000 - val_auc: 0.5667 - val_loss: 0.6986 - learning_rate: 0.0010
Epoch 5/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step - accuracy: 0.7857 - auc: 0.8123 - loss: 0.5103 - val_accuracy: 0.5000 - val_auc: 0.7333 - val_loss: 0.7068 - learning_rate: 0.0010
Epoch 6/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 259ms/step - accuracy: 0.6250


✅ CNN training complete — models saved


## 📊 Cell 14: CNN Training History Plots

In [17]:
def plot_training_history(dtype):
    history = cnn_histories[dtype]
    h = history.history

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'CNN Training History — {dtype.upper()}',
                 fontsize=14, fontweight='bold')

    metrics_pairs = [
        ('accuracy', 'val_accuracy', 'Accuracy', '#2196F3'),
        ('loss',     'val_loss',     'Loss',     '#FF5722'),
        ('auc',      'val_auc',      'AUC',      '#4CAF50'),
    ]

    for ax, (train_key, val_key, title, color) in zip(axes, metrics_pairs):
        if train_key not in h:
            continue
        epochs_r = range(1, len(h[train_key]) + 1)
        ax.plot(epochs_r, h[train_key],     color=color,     lw=2,   label='Train')
        ax.plot(epochs_r, h[val_key],       color=color,     lw=2,   label='Val',
                linestyle='--')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = PLOT_DIR / f'cnn_training_{dtype}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 Saved → {save_path}')

for dt in DRAWING_TYPES:
    plot_training_history(dt)

💾 Saved → outputs\plots\cnn_training_spiral.png
💾 Saved → outputs\plots\cnn_training_wave.png


## 📉 Cell 15: CNN ROC-AUC & Confusion Matrix

In [18]:
def evaluate_cnn(dtype):
    model = cnn_models[dtype]
    X_test, y_test = cnn_tests[dtype]

    y_proba = model.predict(X_test, verbose=0).ravel()
    y_pred  = (y_proba >= 0.5).astype(int)

    metrics = {
        'accuracy':  accuracy_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_proba),
        'mcc':       matthews_corrcoef(y_test, y_pred),
    }

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'CNN Evaluation — {dtype.upper()}',
                 fontsize=14, fontweight='bold')

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[0],
                xticklabels=['Healthy','Parkinson'],
                yticklabels=['Healthy','Parkinson'],
                linewidths=1, linecolor='white')
    axes[0].set_title(f'Confusion Matrix\nAcc={metrics["accuracy"]:.3f}', fontweight='bold')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('True')

    # ROC
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    axes[1].plot(fpr, tpr, color='#4CAF50', lw=2.5,
                 label=f'CNN (AUC={metrics["roc_auc"]:.3f})')
    axes[1].plot([0,1],[0,1],'k--', lw=1)
    axes[1].set_xlabel('FPR')
    axes[1].set_ylabel('TPR')
    axes[1].set_title('ROC-AUC Curve', fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = PLOT_DIR / f'cnn_evaluation_{dtype}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 Saved → {save_path}')

    return metrics, y_test, y_proba


cnn_eval = {}
for dt in DRAWING_TYPES:
    metrics, y_test, y_proba = evaluate_cnn(dt)
    cnn_eval[dt] = (metrics, y_test, y_proba)
    print(f'\n{dt.upper()} CNN Metrics:')
    for k, v in metrics.items():
        print(f'  {k:<12}: {v:.4f}')

💾 Saved → outputs\plots\cnn_evaluation_spiral.png

SPIRAL CNN Metrics:
  accuracy    : 0.5000
  f1          : 0.0000
  recall      : 0.0000
  precision   : 0.0000
  roc_auc     : 0.6133
  mcc         : 0.0000
💾 Saved → outputs\plots\cnn_evaluation_wave.png

WAVE CNN Metrics:
  accuracy    : 0.5000
  f1          : 0.0000
  recall      : 0.0000
  precision   : 0.0000
  roc_auc     : 0.7600
  mcc         : 0.0000


## 🔥 Cell 16: Grad-CAM Explainability

In [19]:
class GradCAM:
    """Gradient-weighted Class Activation Maps."""

    def __init__(self, model, last_conv_layer_name='conv3b'):
        self.model = model
        self.grad_model = Model(
            inputs=model.inputs,
            outputs=[model.get_layer(last_conv_layer_name).output, model.output]
        )

    def compute(self, img_array):
        """img_array: (1, H, W, C) float32"""
        with tf.GradientTape() as tape:
            conv_outputs, predictions = self.grad_model(img_array)
            loss = predictions[:, 0]

        grads = tape.gradient(loss, conv_outputs)
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

        conv_outputs = conv_outputs[0]
        heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
        heatmap = tf.squeeze(heatmap)
        heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
        return heatmap.numpy(), float(predictions[0][0])

    def overlay(self, img_array, heatmap, alpha=0.4):
        """Overlay heatmap onto original image."""
        img   = (img_array[0] * 255).astype(np.uint8)
        h, w  = img.shape[:2]
        heat  = cv2.resize(heatmap, (w, h))
        heat  = np.uint8(255 * heat)
        heat_col = cv2.applyColorMap(heat, cv2.COLORMAP_JET)
        heat_col = cv2.cvtColor(heat_col, cv2.COLOR_BGR2RGB)
        overlaid = cv2.addWeighted(img, 1-alpha, heat_col, alpha, 0)
        return overlaid


def plot_gradcam(dtype, n_samples=4):
    model  = cnn_models[dtype]
    gradcam = GradCAM(model, last_conv_layer_name='conv3b')

    paths = get_paths(dtype)
    healthy_paths   = paths['test_healthy'][:n_samples//2]
    parkinson_paths = paths['test_parkinson'][:n_samples//2]
    all_paths  = healthy_paths + parkinson_paths
    all_labels = ['Healthy']*(n_samples//2) + ['Parkinson']*(n_samples//2)

    fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4*n_samples))
    fig.suptitle(f'Grad-CAM Explainability — {dtype.upper()}',
                 fontsize=14, fontweight='bold')
    col_titles = ['Original', 'Heatmap', 'Overlay']
    for j, ct in enumerate(col_titles):
        axes[0][j].set_title(ct, fontsize=12, fontweight='bold')

    for i, (path, true_label) in enumerate(zip(all_paths, all_labels)):
        arr = preproc.normalize_for_cnn(path)[np.newaxis]
        heatmap, pd_prob = gradcam.compute(arr)
        overlay = gradcam.overlay(arr, heatmap)

        heat_rgb = cv2.applyColorMap(
            np.uint8(255 * cv2.resize(heatmap, IMG_SIZE)), cv2.COLORMAP_JET)
        heat_rgb = cv2.cvtColor(heat_rgb, cv2.COLOR_BGR2RGB)

        axes[i][0].imshow(arr[0])
        axes[i][0].set_ylabel(
            f'True: {true_label}\nPD Prob: {pd_prob:.2f}', fontsize=9)
        axes[i][1].imshow(heat_rgb)
        axes[i][2].imshow(overlay)
        for j in range(3):
            axes[i][j].axis('off')

    plt.tight_layout()
    save_path = PLOT_DIR / f'gradcam_{dtype}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 Saved → {save_path}')

for dt in DRAWING_TYPES:
    plot_gradcam(dt)

💾 Saved → outputs\plots\gradcam_spiral.png
💾 Saved → outputs\plots\gradcam_wave.png


## 🩺 Cell 17: PD Probability & Tremor Severity Index

In [20]:
def compute_tremor_severity(gray):
    """
    Tremor Severity Index (TSI) in [0, 1].
    Combines:
      - High-frequency power ratio
      - Contour irregularity
      - Edge density
    """
    feats = feat_extractor.tremor_frequency(gray)
    tremor_index = feats[2]  # high-freq ratio

    edges = preproc.edge_detect(gray)
    edge_density = edges.sum() / (edges.size * 255)

    # Combine (weights tuned empirically)
    tsi = 0.5 * tremor_index + 0.3 * edge_density + 0.2 * feats[1] / 10.0
    return float(np.clip(tsi, 0, 1))


def analyse_drawing(image_path, dtype='spiral'):
    """
    Full analysis of a single drawing.
    Returns: pd_probability, tremor_severity_index, severity_label
    """
    model = cnn_models[dtype]
    arr = preproc.normalize_for_cnn(image_path)[np.newaxis]
    pd_prob = float(model.predict(arr, verbose=0)[0][0])

    result = preproc.process(image_path)
    tsi = compute_tremor_severity(result['denoised'])

    if tsi < 0.2:   severity = 'None / Very Mild'
    elif tsi < 0.4: severity = 'Mild'
    elif tsi < 0.6: severity = 'Moderate'
    elif tsi < 0.8: severity = 'Severe'
    else:           severity = 'Very Severe'

    return pd_prob, tsi, severity


# Demo on a few test samples
print('=== PD Probability & Tremor Severity Demo ===\n')
for dt in DRAWING_TYPES:
    paths = get_paths(dt)
    print(f'  ── {dt.upper()} ──')
    for label_name, path_list in [('Healthy', paths['test_healthy']),
                                   ('Parkinson', paths['test_parkinson'])]:
        p = path_list[0]
        pd_prob, tsi, severity = analyse_drawing(p, dt)
        print(f'    [{label_name}] {p.name}')
        print(f'      PD Probability   : {pd_prob:.4f}')
        print(f'      Tremor Severity  : {tsi:.4f} ({severity})')
    print()

=== PD Probability & Tremor Severity Demo ===

  ── SPIRAL ──
    [Healthy] V01HE01.png
      PD Probability   : 0.4949
      Tremor Severity  : 0.3999 (Mild)
    [Parkinson] V01PE01.png
      PD Probability   : 0.4947
      Tremor Severity  : 0.1794 (None / Very Mild)

  ── WAVE ──
    [Healthy] V01HO01.png
      PD Probability   : 0.4166
      Tremor Severity  : 0.1150 (None / Very Mild)
    [Parkinson] V01PO01.png
      PD Probability   : 0.4171
      Tremor Severity  : 0.1140 (None / Very Mild)



## 📊 Cell 18: Comprehensive Metrics Summary Table

In [21]:
rows = []
for dt in DRAWING_TYPES:
    # Classical
    res, y_test = classical_results[dt]
    for name, metrics in res.items():
        rows.append({
            'Drawing Type': dt.capitalize(),
            'Model': name,
            'Accuracy':  round(metrics['accuracy'],  4),
            'Precision': round(metrics['precision'], 4),
            'Recall':    round(metrics['recall'],    4),
            'F1-Score':  round(metrics['f1'],        4),
            'ROC-AUC':   round(metrics['roc_auc'],   4),
            'MCC':       round(metrics['mcc'],       4),
        })

    # CNN
    cnn_metrics, _, _ = cnn_eval[dt]
    rows.append({
        'Drawing Type': dt.capitalize(),
        'Model': 'CNN (Deep Learning)',
        'Accuracy':  round(cnn_metrics['accuracy'],  4),
        'Precision': round(cnn_metrics['precision'], 4),
        'Recall':    round(cnn_metrics['recall'],    4),
        'F1-Score':  round(cnn_metrics['f1'],        4),
        'ROC-AUC':   round(cnn_metrics['roc_auc'],   4),
        'MCC':       round(cnn_metrics['mcc'],       4),
    })

df_metrics = pd.DataFrame(rows)
df_metrics.to_csv(OUTPUT_DIR / 'all_metrics.csv', index=False)
print('All Metrics Summary:')
print(df_metrics.to_string(index=False))
print(f'\n💾 Saved → {OUTPUT_DIR}/all_metrics.csv')

All Metrics Summary:
Drawing Type               Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC    MCC
      Spiral           SVM (RBF)    0.7333     0.7692  0.6667    0.7143   0.8800 0.4709
      Spiral       Random Forest    0.8333     0.8125  0.8667    0.8387   0.9111 0.6682
      Spiral CNN (Deep Learning)    0.5000     0.0000  0.0000    0.0000   0.6133 0.0000
        Wave           SVM (RBF)    0.7333     0.6842  0.8667    0.7647   0.8933 0.4842
        Wave       Random Forest    0.7000     0.6875  0.7333    0.7097   0.8489 0.4009
        Wave CNN (Deep Learning)    0.5000     0.0000  0.0000    0.0000   0.7600 0.0000

💾 Saved → outputs/all_metrics.csv


## 📊 Cell 19: Metrics Bar Chart Comparison

In [22]:
def plot_metrics_comparison():
    metric_cols = ['Accuracy','Precision','Recall','F1-Score','ROC-AUC']

    fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)
    fig.suptitle('Model Performance Comparison', fontsize=15, fontweight='bold')
    palette = ['#2196F3', '#FF5722', '#4CAF50']

    for ax, dt in zip(axes, DRAWING_TYPES):
        sub = df_metrics[df_metrics['Drawing Type'] == dt.capitalize()]
        x   = np.arange(len(metric_cols))
        bar_w = 0.25

        for i, (_, row) in enumerate(sub.iterrows()):
            bars = ax.bar(x + i*bar_w,
                          [row[m] for m in metric_cols],
                          bar_w, label=row['Model'],
                          color=palette[i % len(palette)], alpha=0.85)
            for bar in bars:
                ax.text(bar.get_x() + bar.get_width()/2,
                        bar.get_height() + 0.005,
                        f'{bar.get_height():.2f}',
                        ha='center', va='bottom', fontsize=7)

        ax.set_title(f'{dt.upper()} Drawings', fontweight='bold', fontsize=12)
        ax.set_xticks(x + bar_w)
        ax.set_xticklabels(metric_cols, rotation=20)
        ax.set_ylim(0, 1.12)
        ax.set_ylabel('Score')
        ax.legend(fontsize=8)
        ax.grid(True, axis='y', alpha=0.3)

    plt.tight_layout()
    save_path = PLOT_DIR / 'metrics_comparison.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 Saved → {save_path}')

plot_metrics_comparison()

💾 Saved → outputs\plots\metrics_comparison.png


## 📊 Cell 20: Combined ROC Comparison (All Models)

In [23]:
def plot_combined_roc():
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle('Combined ROC-AUC — All Models', fontsize=14, fontweight='bold')
    line_styles = ['-', '--', '-.']
    line_colors = ['#2196F3', '#FF5722', '#4CAF50']

    for ax, dt in zip(axes, DRAWING_TYPES):
        ax.plot([0,1],[0,1],'k:', lw=1, label='Random')

        # Classical
        res, y_test = classical_results[dt]
        for (name, metrics), ls, color in zip(res.items(), line_styles, line_colors):
            fpr, tpr, _ = roc_curve(y_test, metrics['y_proba'])
            ax.plot(fpr, tpr, ls=ls, color=color, lw=2.2,
                    label=f'{name} (AUC={metrics["roc_auc"]:.3f})')

        # CNN
        cnn_metrics, y_test_cnn, y_proba_cnn = cnn_eval[dt]
        fpr, tpr, _ = roc_curve(y_test_cnn, y_proba_cnn)
        ax.plot(fpr, tpr, ls='-', color='#9C27B0', lw=2.5,
                label=f'CNN (AUC={cnn_metrics["roc_auc"]:.3f})')

        ax.set_title(f'{dt.upper()} Drawings', fontweight='bold', fontsize=12)
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.legend(fontsize=9, loc='lower right')
        ax.grid(True, alpha=0.3)
        ax.set_xlim([-0.01, 1.01])
        ax.set_ylim([-0.01, 1.01])

    plt.tight_layout()
    save_path = PLOT_DIR / 'combined_roc_auc.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 Saved → {save_path}')

plot_combined_roc()

💾 Saved → outputs\plots\combined_roc_auc.png


## 🎯 Cell 21: PD Probability Distribution Plot

In [24]:
def plot_pd_probability_distribution(dtype):
    model = cnn_models[dtype]
    paths = get_paths(dtype)

    healthy_probs, parkinson_probs = [], []
    for p in paths['test_healthy']:
        try:
            arr = preproc.normalize_for_cnn(p)[np.newaxis]
            healthy_probs.append(float(model.predict(arr, verbose=0)[0][0]))
        except: pass
    for p in paths['test_parkinson']:
        try:
            arr = preproc.normalize_for_cnn(p)[np.newaxis]
            parkinson_probs.append(float(model.predict(arr, verbose=0)[0][0]))
        except: pass

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(healthy_probs,   bins=10, alpha=0.7, color='steelblue', label='Healthy',
            edgecolor='white')
    ax.hist(parkinson_probs, bins=10, alpha=0.7, color='tomato',    label='Parkinson',
            edgecolor='white')
    ax.axvline(0.5, color='black', linestyle='--', lw=2, label='Decision Boundary (0.5)')
    ax.set_xlabel('PD Probability', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title(f'PD Probability Distribution — CNN {dtype.upper()}',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

    save_path = PLOT_DIR / f'pd_prob_distribution_{dtype}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 Saved → {save_path}')

for dt in DRAWING_TYPES:
    plot_pd_probability_distribution(dt)

💾 Saved → outputs\plots\pd_prob_distribution_spiral.png
💾 Saved → outputs\plots\pd_prob_distribution_wave.png


## 🌡️ Cell 22: Tremor Severity Heatmap

In [25]:
def plot_tremor_severity_comparison(dtype):
    paths = get_paths(dtype)

    h_tsi, p_tsi = [], []
    for p in paths['test_healthy']:
        try:
            r = preproc.process(p)
            h_tsi.append(compute_tremor_severity(r['denoised']))
        except: pass
    for p in paths['test_parkinson']:
        try:
            r = preproc.process(p)
            p_tsi.append(compute_tremor_severity(r['denoised']))
        except: pass

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Tremor Severity Index — {dtype.upper()}',
                 fontsize=14, fontweight='bold')

    # Violin plot
    all_data   = h_tsi + p_tsi
    all_labels = ['Healthy']*len(h_tsi) + ['Parkinson']*len(p_tsi)
    df_tsi = pd.DataFrame({'TSI': all_data, 'Class': all_labels})
    sns.violinplot(data=df_tsi, x='Class', y='TSI', palette=['steelblue','tomato'],
                   ax=axes[0], inner='box')
    axes[0].set_title('TSI Distribution', fontweight='bold')
    axes[0].grid(True, alpha=0.3)

    # Bar chart per image
    n_show = min(10, len(h_tsi), len(p_tsi))
    x = np.arange(n_show)
    axes[1].bar(x - 0.2, h_tsi[:n_show], 0.4, color='steelblue', label='Healthy', alpha=0.85)
    axes[1].bar(x + 0.2, p_tsi[:n_show], 0.4, color='tomato',    label='Parkinson', alpha=0.85)
    axes[1].set_xlabel('Sample Index')
    axes[1].set_ylabel('Tremor Severity Index')
    axes[1].set_title('Per-Sample TSI (first 10 test)', fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = PLOT_DIR / f'tremor_severity_{dtype}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 Saved → {save_path}')

for dt in DRAWING_TYPES:
    plot_tremor_severity_comparison(dt)

💾 Saved → outputs\plots\tremor_severity_spiral.png
💾 Saved → outputs\plots\tremor_severity_wave.png


## 🧩 Cell 23: Feature Correlation Heatmap

In [26]:
def plot_feature_correlation(dtype):
    X_train, y_train, _, _, _ = feature_data[dtype]
    df_feat = pd.DataFrame(X_train, columns=FEATURE_LABELS)
    df_feat['Label'] = y_train

    corr = df_feat.drop('Label', axis=1).corr()

    fig, ax = plt.subplots(figsize=(16, 13))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0,
                vmin=-1, vmax=1, ax=ax, linewidths=0.3,
                xticklabels=True, yticklabels=True)
    ax.set_title(f'Feature Correlation Matrix — {dtype.upper()}',
                 fontsize=13, fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(rotation=0, fontsize=8)

    plt.tight_layout()
    save_path = PLOT_DIR / f'feature_correlation_{dtype}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 Saved → {save_path}')

for dt in DRAWING_TYPES:
    plot_feature_correlation(dt)

💾 Saved → outputs\plots\feature_correlation_spiral.png
💾 Saved → outputs\plots\feature_correlation_wave.png


## 🎯 Cell 24: Random Forest Feature Importance

In [27]:
def plot_rf_feature_importance(dtype):
    res, _ = classical_results[dtype]
    rf = res['Random Forest']['clf']

    importances = rf.feature_importances_
    indices     = np.argsort(importances)[::-1][:20]  # top 20

    fig, ax = plt.subplots(figsize=(12, 6))
    colors = plt.cm.RdYlGn(importances[indices] / importances[indices].max())
    bars = ax.bar(range(20), importances[indices], color=colors)
    ax.set_xticks(range(20))
    ax.set_xticklabels([FEATURE_LABELS[i] for i in indices],
                        rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Importance', fontsize=11)
    ax.set_title(f'Top-20 Feature Importances — Random Forest {dtype.upper()}',
                 fontsize=13, fontweight='bold')
    ax.grid(True, axis='y', alpha=0.3)

    plt.tight_layout()
    save_path = PLOT_DIR / f'rf_feature_importance_{dtype}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 Saved → {save_path}')

for dt in DRAWING_TYPES:
    plot_rf_feature_importance(dt)

💾 Saved → outputs\plots\rf_feature_importance_spiral.png
💾 Saved → outputs\plots\rf_feature_importance_wave.png


## 💾 Cell 25: Save Feature Extractor & Final Summary

In [28]:
# Save feature extractor
joblib.dump(feat_extractor, MODEL_DIR / 'feature_extractor.pkl')
print('💾 Feature extractor saved →', MODEL_DIR / 'feature_extractor.pkl')

# Save preprocessing parameters
preproc_config = {
    'target_size': IMG_SIZE,
    'img_size': IMG_SIZE,
}
joblib.dump(preproc_config, MODEL_DIR / 'preproc_config.pkl')

# Final summary
print('\n' + '='*60)
print('  FINAL RESULTS SUMMARY')
print('='*60)
print(df_metrics.to_string(index=False))

print('\n📂 Saved Files:')
print('\n  Models:')
for f in sorted(MODEL_DIR.glob('*')):
    print(f'    {f.name}')

print('\n  Plots:')
for f in sorted(PLOT_DIR.glob('*.png')):
    print(f'    {f.name}')

print('\n  Features:')
for f in sorted(FEATURE_DIR.glob('*')):
    print(f'    {f.name}')

print('\n✅ Pipeline complete!')

💾 Feature extractor saved → outputs\models\feature_extractor.pkl

  FINAL RESULTS SUMMARY
Drawing Type               Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC    MCC
      Spiral           SVM (RBF)    0.7333     0.7692  0.6667    0.7143   0.8800 0.4709
      Spiral       Random Forest    0.8333     0.8125  0.8667    0.8387   0.9111 0.6682
      Spiral CNN (Deep Learning)    0.5000     0.0000  0.0000    0.0000   0.6133 0.0000
        Wave           SVM (RBF)    0.7333     0.6842  0.8667    0.7647   0.8933 0.4842
        Wave       Random Forest    0.7000     0.6875  0.7333    0.7097   0.8489 0.4009
        Wave CNN (Deep Learning)    0.5000     0.0000  0.0000    0.0000   0.7600 0.0000

📂 Saved Files:

  Models:
    cnn_spiral.h5
    cnn_spiral_best.keras
    cnn_wave.h5
    cnn_wave_best.keras
    feature_extractor.pkl
    preproc_config.pkl
    random_spiral.pkl
    random_wave.pkl
    svm_spiral.pkl
    svm_wave.pkl

  Plots:
    cnn_evaluation_spiral.png
    cnn_evaluati

## 🔮 Cell 26: Inference on Single New Image

In [29]:
def predict_single(image_path, dtype='spiral', visualise=True):
    """
    Run full analysis on a single new image.

    Parameters
    ----------
    image_path : str | Path
    dtype      : 'spiral' or 'wave'
    visualise  : show Grad-CAM overlay

    Returns
    -------
    dict with pd_probability, tremor_severity_index, severity_label,
         classical_svm_prob, classical_rf_prob
    """
    image_path = Path(image_path)
    assert image_path.exists(), f'File not found: {image_path}'

    # ── CNN
    model = cnn_models[dtype]
    arr   = preproc.normalize_for_cnn(image_path)[np.newaxis]
    pd_prob_cnn = float(model.predict(arr, verbose=0)[0][0])

    # ── Classical
    feats  = feat_extractor.extract(image_path)
    scaler = joblib.load(FEATURE_DIR / f'scaler_{dtype}.pkl')
    feats_sc = scaler.transform(feats.reshape(1,-1))

    svm = joblib.load(MODEL_DIR / f'svm_{dtype}.pkl')
    rf  = joblib.load(MODEL_DIR / f'random_{dtype}.pkl')
    pd_prob_svm = float(svm.predict_proba(feats_sc)[0][1])
    pd_prob_rf  = float(rf.predict_proba(feats_sc)[0][1])

    # Ensemble
    pd_prob_ensemble = (pd_prob_cnn + pd_prob_svm + pd_prob_rf) / 3.0

    # ── TSI
    result = preproc.process(image_path)
    tsi = compute_tremor_severity(result['denoised'])
    if tsi < 0.2:   severity = 'None / Very Mild'
    elif tsi < 0.4: severity = 'Mild'
    elif tsi < 0.6: severity = 'Moderate'
    elif tsi < 0.8: severity = 'Severe'
    else:           severity = 'Very Severe'

    # ── Visualise
    if visualise:
        gradcam = GradCAM(model, 'conv3b')
        heatmap, _ = gradcam.compute(arr)
        overlay = gradcam.overlay(arr, heatmap)

        fig, axes = plt.subplots(1, 3, figsize=(14, 5))
        fig.suptitle(f'Analysis: {image_path.name}', fontsize=13, fontweight='bold')
        axes[0].imshow(arr[0]);      axes[0].set_title('Input')
        heat_rgb = cv2.applyColorMap(
            np.uint8(255*cv2.resize(heatmap, IMG_SIZE)), cv2.COLORMAP_JET)
        heat_rgb = cv2.cvtColor(heat_rgb, cv2.COLOR_BGR2RGB)
        axes[1].imshow(heat_rgb);    axes[1].set_title('Grad-CAM')
        axes[2].imshow(overlay);     axes[2].set_title('Overlay')
        for ax in axes: ax.axis('off')

        info  = (f'PD Probability (CNN): {pd_prob_cnn:.3f}\n'
                 f'PD Probability (SVM): {pd_prob_svm:.3f}\n'
                 f'PD Probability (RF):  {pd_prob_rf:.3f}\n'
                 f'Ensemble Probability: {pd_prob_ensemble:.3f}\n'
                 f'Tremor Severity:      {tsi:.3f} ({severity})')
        fig.text(0.5, -0.02, info, ha='center', fontsize=11, fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
        plt.tight_layout()
        plt.show()

    return {
        'pd_probability_cnn':      pd_prob_cnn,
        'pd_probability_svm':      pd_prob_svm,
        'pd_probability_rf':       pd_prob_rf,
        'pd_probability_ensemble': pd_prob_ensemble,
        'tremor_severity_index':   tsi,
        'severity_label':          severity,
    }


# ── Example Usage (change path to any image you want to test) ──────────────
# result = predict_single('drawings/spiral/testing/parkinson/image1.png', dtype='spiral')
# print(result)

# Demo on first test image
sample = get_paths('spiral')['test_parkinson'][0]
result = predict_single(sample, dtype='spiral', visualise=True)
print('\nPrediction output:')
for k, v in result.items():
    print(f'  {k:<32}: {v}')


Prediction output:
  pd_probability_cnn              : 0.4946853220462799
  pd_probability_svm              : 0.9929923892691085
  pd_probability_rf               : 0.865
  pd_probability_ensemble         : 0.7842259037717962
  tremor_severity_index           : 0.17943912297487258
  severity_label                  : None / Very Mild
